# components.scrollbar

> Custom scrollbar component with proportional thumb for position indication.

In [ ]:
#| default_exp components.scrollbar

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fasthtml.common import Div

from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.layout import position
from cjm_fasthtml_tailwind.utilities.interactivity import cursor, select
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionConfig, VirtualCollectionState
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.windowing import compute_scrollbar

In [ ]:
#| export
def render_scrollbar_thumb(
    state: VirtualCollectionState,       # Collection state
    config: VirtualCollectionConfig,      # Collection config
    ids: VirtualCollectionHtmlIds,        # HTML IDs
    track_height: float = 600.0,          # Track height for min thumb calculation
    oob: bool = False,                    # Whether to include hx-swap-oob
) -> Div:  # Thumb element
    """Render the scrollbar thumb at the correct position."""
    thumb_top, thumb_height = compute_scrollbar(
        state.window_start, state.visible_rows, state.total_items,
        track_height, config.min_thumb_height,
    )

    thumb = Div(
        id=ids.scrollbar_thumb,
        cls=combine_classes(
            position.absolute, w.full, border_radius.field,
            bg_dui.base_content.opacity(30),
            cursor.grab,
        ),
        style=f"top: {thumb_top:.2f}%; height: {thumb_height:.2f}%",
    )

    if oob:
        thumb.attrs["hx-swap-oob"] = "outerHTML"

    return thumb

In [ ]:
#| export
def render_scrollbar(
    state: VirtualCollectionState,       # Collection state
    config: VirtualCollectionConfig,      # Collection config
    ids: VirtualCollectionHtmlIds,        # HTML IDs
) -> Div:  # Complete scrollbar (track + thumb)
    """Render the custom scrollbar with track and proportional thumb."""
    # Don't show if all items visible
    if state.total_items <= state.visible_rows:
        return Div(id=ids.scrollbar_track)

    thumb = render_scrollbar_thumb(state, config, ids)

    return Div(
        thumb,
        id=ids.scrollbar_track,
        cls=combine_classes(
            w(2), position.relative, border_radius.field,
            bg_dui.base_content.opacity(10),
            cursor.pointer, select.none,
        ),
        data_total_items=str(state.total_items),
        data_visible_rows=str(state.visible_rows),
    )

## Tests

In [ ]:
from fasthtml.common import to_xml
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionConfig, VirtualCollectionState
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

config = VirtualCollectionConfig(prefix="t", min_thumb_height=24)
ids = VirtualCollectionHtmlIds(prefix="t")

In [ ]:
# Test scrollbar at start
state = VirtualCollectionState(total_items=500, visible_rows=15, window_start=0)
sb = render_scrollbar(state, config, ids)
html = to_xml(sb)
assert 'id="t-scrollbar-track"' in html
assert 'id="t-scrollbar-thumb"' in html
assert 'top: 0.00%' in html
assert 'cursor-grab' in html
print("Scrollbar at start test passed")

Scrollbar at start test passed


In [ ]:
# Test scrollbar at end
state = VirtualCollectionState(total_items=500, visible_rows=15, window_start=485)
sb = render_scrollbar(state, config, ids)
html = to_xml(sb)
# Thumb should be near the bottom
assert 'id="t-scrollbar-thumb"' in html
print("Scrollbar at end test passed")

Scrollbar at end test passed


In [ ]:
# Test scrollbar hidden when all items visible
state = VirtualCollectionState(total_items=10, visible_rows=15, window_start=0)
sb = render_scrollbar(state, config, ids)
html = to_xml(sb)
assert 'id="t-scrollbar-track"' in html
assert 'id="t-scrollbar-thumb"' not in html  # No thumb when all visible
print("Scrollbar hidden test passed")

Scrollbar hidden test passed


In [ ]:
# Test OOB thumb
state = VirtualCollectionState(total_items=500, visible_rows=15, window_start=100)
thumb = render_scrollbar_thumb(state, config, ids, oob=True)
html = to_xml(thumb)
assert 'hx-swap-oob="outerHTML"' in html
assert 'id="t-scrollbar-thumb"' in html
print("OOB thumb test passed")

OOB thumb test passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()